# Step 3: Automatic Iterative Matching
TPS-based iterative matching with classifier gating.

Inputs: initial landmarks from step_2, trained classifier from step_0.
Output: `{subject}_{date}_landmarks_auto_final.csv` + extended coreg table.

In [ ]:
import sys, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path('.')))

from coreg_data_loading import (
    find_coreg_dirs, parse_coreg_dir_name, load_subject_data,
    load_landmarks, save_landmarks
)
from coreg_matching import run_auto_matching, fit_tps, project_czstack_to_hcr
from coreg_alignment import CZ_RESOLUTION_UM

%load_ext autoreload
%autoreload 2

DATA_DIR = Path('/root/capsule/data')
SCRATCH_DIR = Path('/root/capsule/scratch')
CLASSIFIER_PATH = DATA_DIR / 'match_classifier.pkl'


## Parameters

In [ ]:
# ── User parameters ──────────────────────────────────────────────────────────
subject_id = '790322'
czstack_date = '2025-07-10'

gfp_threshold_counts = 5

matching_params = {
    'max_iterations': 10,
    'k_neighbors': 5,
    'hcr_value_feature': 'density',
    'filter_rate_from_min': 1.2,
    'accept_probability_threshold': 0.6,   # ignored if no classifier
    'sampling_threshold': 500,
    'patch_sizes_um': [20, 50, 100],
    'min_neighbors_for_scale': 3,
    'distance_threshold_um': 30.0,
    'convergence_rate': 0.005,
}


## Load Data & Classifier

In [ ]:
coreg_dirs = find_coreg_dirs(DATA_DIR)
coreg_dir = None
for d in coreg_dirs:
    sid, cdate = parse_coreg_dir_name(d)
    if sid == subject_id and cdate == czstack_date:
        coreg_dir = d
        break

save_dir = SCRATCH_DIR / f'{subject_id}_{czstack_date}_coreg_cpsam'
save_dir.mkdir(parents=True, exist_ok=True)
if coreg_dir is None:
    coreg_dir = save_dir

data = load_subject_data(coreg_dir, subject_id, czstack_date,
                          gfp_threshold=gfp_threshold_counts, load_iter_paths=True)
czstack_df = data['czstack_df']
hcr_df = data['hcr_df']
hcr_scales = data['hcr_scales']
hcr_metrics = data['hcr_metrics']
spot_counts = data['spot_counts']

print(f'Czstack: {len(czstack_df)} cells')
print(f'HCR: {len(hcr_df)} cells total, '
      f'{spot_counts["is_gfp"].sum()} GFP+ (threshold={gfp_threshold_counts})')


In [ ]:
# Load classifier (optional – falls back to MNN-only if absent)
classifier = None
if CLASSIFIER_PATH.exists():
    with open(CLASSIFIER_PATH, 'rb') as f:
        classifier = pickle.load(f)
    print(f'Classifier loaded from {CLASSIFIER_PATH}')
else:
    print('WARNING: No classifier found. Will use MNN + distance threshold only.')


## Load Initial Landmarks (from step_2)

In [ ]:
initial_lm_path = save_dir / f'{subject_id}_{czstack_date}_landmarks_initial.csv'

if not initial_lm_path.exists():
    # Fall back to manual landmarks if auto-initial not available
    fallback_lm_files = sorted(coreg_dir.glob(f'{subject_id}*landmarks.csv'))
    if fallback_lm_files:
        initial_lm_path = fallback_lm_files[0]
        print(f'Using manual initial landmarks: {initial_lm_path.name}')
    else:
        raise FileNotFoundError(
            f'No initial landmarks found in {save_dir} or {coreg_dir}. '
            'Please run step_2_initial_alignment.ipynb first.'
        )

seed_landmarks = load_landmarks(initial_lm_path)
print(f'Seed landmarks: {len(seed_landmarks)} ({seed_landmarks["active"].sum()} active)')


## Load Czstack Volume (for NCC features)

In [ ]:
# Optional: load czstack volume for NCC-based features
# Skip if not available (NCC features will be NaN, classifier falls back to geometric+constellation)
czstack_vol = None
hcr_zarr_path = None

try:
    fps = data['filepaths']
    cz_tif = fps.get('czstack_reg_dim_swapped_path', fps.get('czstack_reg_path'))
    if cz_tif and Path(cz_tif).exists():
        import tifffile
        print(f'Loading czstack volume from {Path(cz_tif).name}...')
        czstack_vol = tifffile.imread(str(cz_tif))
        print(f'  Shape: {czstack_vol.shape}, dtype: {czstack_vol.dtype}')
        if czstack_vol.ndim == 4:
            czstack_vol = czstack_vol[0]  # drop channel dim if present
    else:
        print('Czstack volume not found, NCC features will be NaN')

    hcr_dir = fps['hcr_centroid_path'].parent.parent
    zarr_path = hcr_dir / 'image_tile_fusing' / 'fused' / 'channel_488.zarr'
    if zarr_path.exists():
        hcr_zarr_path = str(zarr_path)
        print(f'HCR zarr path: {zarr_path.name}')
    else:
        print('HCR zarr not found, NCC features will be NaN')
except Exception as e:
    print(f'Could not load volumes: {e}')


## Run Automatic Matching

In [ ]:
print('Starting automatic iterative matching...')

all_matches = run_auto_matching(
    seed_landmarks=seed_landmarks,
    czstack_df=czstack_df,
    hcr_df=hcr_df,
    spot_counts=spot_counts,
    hcr_metrics=hcr_metrics,
    hcr_scales=hcr_scales,
    czstack_vol=czstack_vol,
    hcr_zarr_path=hcr_zarr_path,
    classifier=classifier,
    czstack_res_um=CZ_RESOLUTION_UM,
    params=matching_params,
)

n_seed = seed_landmarks['active'].sum()
n_total_matched = len(all_matches) + n_seed
n_cz = len(czstack_df)
print(f'\n=== Matching Complete ===')
print(f'Seed matches: {n_seed}')
print(f'Auto-matched: {len(all_matches)}')
print(f'Total: {n_total_matched}/{n_cz} ({100*n_total_matched/n_cz:.1f}%)')


In [ ]:
# Build final landmark DataFrame combining seeds + auto-matched
from coreg_alignment import CZ_RESOLUTION_UM

final_landmarks = seed_landmarks.copy()

for _, m in all_matches.iterrows():
    cz_id = int(m['czstack_cell_id'])
    hcr_id = int(m['hcr_cell_id'])
    cz_row = czstack_df[czstack_df['czstack_cell_id'] == cz_id]
    hcr_row = hcr_df[hcr_df['hcr_cell_id'] == hcr_id]
    if len(cz_row) == 0 or len(hcr_row) == 0:
        continue
    final_landmarks = pd.concat([final_landmarks, pd.DataFrame([{
        'ids': f'cz{cz_id}-hcr{hcr_id}',
        'active': True,
        'czstack_x': float(cz_row.iloc[0]['czstack_x']),
        'czstack_y': float(cz_row.iloc[0]['czstack_y']),
        'czstack_z': float(cz_row.iloc[0]['czstack_z']),
        'hcr_x': float(hcr_row.iloc[0]['hcr_x']),
        'hcr_y': float(hcr_row.iloc[0]['hcr_y']),
        'hcr_z': float(hcr_row.iloc[0]['hcr_z']),
    }])], ignore_index=True)

final_lm_path = save_dir / f'{subject_id}_{czstack_date}_landmarks_auto_final.csv'
save_landmarks(final_landmarks, final_lm_path)
print(f'Final landmarks saved: {final_lm_path}')
print(f'  Total rows: {len(final_landmarks)} | active: {final_landmarks["active"].sum()}')


In [ ]:
# Save extended match table with per-match metadata
match_meta_path = save_dir / f'{subject_id}_{czstack_date}_auto_match_metadata.csv'
all_matches.to_csv(match_meta_path, index=False)
print(f'Match metadata saved: {match_meta_path}')
print(all_matches.head())


## Visualisation

In [ ]:
if len(all_matches) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Match probability distribution
    if 'match_probability' in all_matches.columns and not all_matches['match_probability'].isna().all():
        axes[0].hist(all_matches['match_probability'].dropna(), bins=40, edgecolor='k')
        axes[0].set_xlabel('Match probability')
        axes[0].set_ylabel('Count')
        axes[0].set_title('Distribution of match probabilities')

    # Cumulative matches per iteration
    if 'iter_matched' in all_matches.columns:
        iters = all_matches.groupby('iter_matched').size().cumsum()
        axes[1].bar(iters.index, iters.values, edgecolor='k')
        axes[1].set_xlabel('Iteration')
        axes[1].set_ylabel('Cumulative matches')
        axes[1].set_title('Cumulative matches per iteration')

    plt.suptitle(f'Auto-matching: {subject_id}')
    plt.tight_layout()
    plt.savefig(save_dir / f'{subject_id}_{czstack_date}_auto_matching_summary.png', dpi=100)
    plt.show()


## QC: Projection Error

In [ ]:
# Compute TPS projection error for matched pairs
from coreg_matching import fit_tps, project_czstack_to_hcr

try:
    tps_final = fit_tps(final_landmarks[final_landmarks['active'] == True])
    hcr_res = np.array([hcr_scales['scale_z'], hcr_scales['scale_y'], hcr_scales['scale_x']])

    matched_cz_ids = all_matches['czstack_cell_id'].values
    matched_cz = czstack_df[czstack_df['czstack_cell_id'].isin(matched_cz_ids)].reset_index(drop=True)
    proj = project_czstack_to_hcr(matched_cz, tps_final)

    matched_hcr_ids = all_matches.set_index('czstack_cell_id').loc[matched_cz['czstack_cell_id'].values, 'hcr_cell_id']
    matched_hcr = hcr_df[hcr_df['hcr_cell_id'].isin(matched_hcr_ids)].set_index('hcr_cell_id').loc[matched_hcr_ids.values]

    true_hcr = matched_hcr[['hcr_z', 'hcr_y', 'hcr_x']].values
    err_um = (proj - true_hcr) * hcr_res
    dist_um = np.linalg.norm(err_um, axis=1)

    print(f'TPS projection errors (µm):')
    print(f'  Median: {np.median(dist_um):.2f}')
    print(f'  Mean: {np.mean(dist_um):.2f}')
    print(f'  95th pct: {np.percentile(dist_um, 95):.2f}')

    plt.figure(figsize=(8, 4))
    plt.hist(dist_um, bins=50, edgecolor='k')
    plt.xlabel('TPS projection error (µm)')
    plt.ylabel('Count')
    plt.title(f'Projection errors: {subject_id}')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Could not compute projection errors: {e}')
